In [1]:
# Installing fastf1 library to access f1 data

!pip install fastf1

   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 1.6/1.6 MB 21.5 MB/s eta 0:00:00


In [26]:
# importing required libraries

import fastf1
import pandas as pd
import os

In [27]:
# creating and enabling cache for api calls

os.makedirs('cache', exist_ok=True)
fastf1.Cache.enable_cache('cache')

In [3]:
years = range(2020,2025) # we will be working with data from 2020-2024

In [13]:
from fastf1.ergast import Ergast
import pandas as pd

ergast = Ergast()
all_circuits = []

for year in range(2020, 2025):
    try:
        response = ergast.get_circuits(season=year, result_type='raw')
        
        for circuit in response:
            circuit_info = {
                'CircuitId': circuit['circuitId'],
                'CircuitName': circuit['circuitName'],
                'Location': circuit['Location']['locality'],
                'Country': circuit['Location']['country']
            }
            all_circuits.append(circuit_info)

    except Exception as e:
        print(f"Failed to fetch circuits for {year}: {e}")


df = pd.DataFrame(all_circuits).drop_duplicates(subset=['CircuitId'])

df.to_csv('circuits.csv', index=False, encoding='utf-8-sig')

print("Saved circuit data to 'circuits.csv'")


Saved circuit data to 'circuits.csv'


In [18]:
from fastf1.ergast import Ergast
from fastf1.plotting import TEAM_COLORS
import pandas as pd

ergast = Ergast()
all_teams = []

# Step 1: Fetch teams from Ergast for each season
for year in range(2020, 2025):
    try:
        response = ergast.get_constructor_info(season=year, result_type='raw')
        
        for team in response:
            team_info = {
                'constructor_ref': team['constructorId'],
                'name': team['name'],
                'nationality': team['nationality']
                # 'team_color' will be added in step 2
            }
            all_teams.append(team_info)

    except Exception as e:
        print(f"Failed to fetch teams for {year}: {e}")

# Step 2: Deduplicate by constructor ID
df_teams = pd.DataFrame(all_teams).drop_duplicates(subset=['constructor_ref'])

# Step 3: Add team color using FastF1 TEAM_COLORS
def get_team_color(name):
    # Try to match FastF1 name keys (case-insensitive)
    for key in TEAM_COLORS:
        if key.lower() in name.lower():
            return TEAM_COLORS[key]
    return None  # fallback if no match found

df_teams['team_color'] = df_teams['name'].apply(get_team_color)

# Step 4: Save to CSV
df_teams.to_csv('teams.csv', index=False, encoding='utf-8-sig')
print("Saved team data with colors to 'teams.csv'")


Saved team data with colors to 'teams.csv'


C:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\fastf1\plotting\__init__.py:84: FutureWarning: INCOMPATIBLE with 2025 season! TEAM_COLORS is deprecated and will be removed in a future version.
  warnings.warn(f"INCOMPATIBLE with 2025 season! {name} is deprecated "


In [20]:
# fetch driver info

from fastf1.ergast import Ergast
import pandas as pd

ergast = Ergast()
all_drivers = []

for year in range(2020, 2025):
    try:
        response = ergast.get_driver_info(season=year, result_type='raw')
        
        for driver in response:
            full_name = f"{driver.get('givenName', '')} {driver.get('familyName', '')}".strip()
            driver_info = {
                'driver_ref': driver['driverId'],
                'full_name': full_name,
                'abbreviation': driver['code'],
                'number': driver["permanentNumber"],
                'dob': driver["dateOfBirth"],
                'nationality': driver['nationality']
            }
            all_drivers.append(driver_info)

    except Exception as e:
        print(f"Failed to fetch drivers for {year}: {e}")

# Keep only one record per driver
df_drivers = pd.DataFrame(all_drivers).drop_duplicates(subset=['driver_ref'])

# Save to CSV
df_drivers.to_csv('drivers.csv', index=False, encoding='utf-8-sig')
print("Saved driver data to 'drivers.csv'")


Saved driver data to 'drivers.csv'


In [5]:
from fastf1.ergast import Ergast
import pandas as pd

ergast = Ergast()
all_events = []

for year in range(2020, 2025):
    try:
        races = ergast.get_race_schedule(season=year, result_type='raw')

        for race in races:
            event_info = {
                'season': year,
                'round': int(race['round']),
                'event_name': race['raceName'],
                'event_date': race['date'],
                'circuit_ref': race['Circuit']['circuitId']
            }
            all_events.append(event_info)

    except Exception as e:
        print(f"Failed to fetch schedule for {year}: {e}")

# Convert to DataFrame
df_events = pd.DataFrame(all_events)

# Save to CSV
df_events.to_csv('events.csv', index=False, encoding='utf-8-sig')
print("Saved raw event data to 'events.csv'")


Saved raw event data to 'events.csv'


In [3]:
# fetching data for driver and team pairing by season

import requests
import pandas as pd

pairings = []

for season in range(2020, 2025):
    for round_num in range(1, 25):  # Up to 23 rounds per season
        url = f"http://ergast.com/api/f1/{season}/{round_num}/results.json"
        response = requests.get(url)

        if response.status_code != 200:
            continue  # skip if round doesn't exist

        races = response.json()['MRData']['RaceTable']['Races']
        if not races:
            continue  # no race data

        results = races[0]['Results']
        for entry in results:
            driver_id = entry['Driver']['driverId']
            constructor_id = entry['Constructor']['constructorId']
            pairings.append({
                'season': season,
                'driver_ref': driver_id,
                'constructor_ref': constructor_id
            })

# Convert to DataFrame and deduplicate by season, driver, constructor
df_pairings = pd.DataFrame(pairings).drop_duplicates()

# Save to CSV
df_pairings.to_csv('driver_team_season.csv', index=False, encoding='utf-8-sig')
print("Saved driver-constructor-season pairings to 'driver_team_season.csv'")



Saved driver-constructor-season pairings to 'driver_team_season.csv'


In [15]:
from fastf1.ergast import Ergast
import pandas as pd
import json

ergast = Ergast()
pairings = []

for season in range(2020, 2025):
    for round_num in range(1, 25):  
        try:
            races = ergast.get_race_results(season=season, round=round_num, result_type='raw')

            if not races:
                continue

            race = races[0] 
            results = race['Results']

            for entry in results:
                driver_id = entry['Driver']['driverId']
                constructor_id = entry['Constructor']['constructorId']
                pairings.append({
                    'season': season,
                    'driver_ref': driver_id,
                    'constructor_ref': constructor_id
                })

        except Exception as e:
            print(f"Failed to fetch results for {season} Round {round_num}: {e}")

# Convert to DataFrame and deduplicate by season, driver, constructor
df_pairings = pd.DataFrame(pairings).drop_duplicates()

# Save to CSV
df_pairings.to_csv('driver_team_season.csv', index=False, encoding='utf-8-sig')
print("Saved driver-constructor-season pairings to 'driver_team_season.csv'")


Saved driver-constructor-season pairings to 'driver_team_season.csv'


In [16]:
# fetching results

from fastf1.ergast import Ergast
import pandas as pd

ergast = Ergast()
results_data = []

for season in range(2020, 2025):
    for round_num in range(1, 25):
        try:
            races = ergast.get_race_results(season=season, round=round_num, result_type='raw')

            if not races:
                continue

            race = races[0]
            result_entries = race['Results']

            for entry in result_entries:
                driver_id = entry['Driver']['driverId']
                constructor_id = entry['Constructor']['constructorId']
                finishing_position = int(entry.get('position', -1))
                grid_position = int(entry.get('grid', -1))
                status = entry.get('status', 'Unknown')
                points = float(entry.get('points', 0.0))

                # Fastest lap fields
                fastest_lap_time = None
                fastest_lap_number = None
                fastest_lap = entry.get('FastestLap')

                if fastest_lap and 'Time' in fastest_lap:
                    fastest_lap_time = fastest_lap['Time']['time']
                    fastest_lap_number = int(fastest_lap['lap'])

                results_data.append({
                    'season': season,
                    'round': round_num,
                    'driver_ref': driver_id,
                    'constructor_ref': constructor_id,
                    'finishing_position': finishing_position,
                    'grid_position': grid_position,
                    'status': status,
                    'points': points,
                    'fastest_lap_time': fastest_lap_time,
                    'fastest_lap_number': fastest_lap_number
                })

        except Exception as e:
            print(f"Failed to fetch results for {season} round {round_num}: {e}")

# Convert to DataFrame
df_results = pd.DataFrame(results_data)

# Save to CSV
df_results.to_csv('results_raw.csv', index=False, encoding='utf-8-sig')
print("Saved race results with constructor_ref to 'results_raw.csv'")


Saved race results with constructor_ref to 'results_raw.csv'


In [37]:
import fastf1
import pandas as pd
import os
import sqlite3
import sys

# Add the path to your project directory if needed
# sys.path.append('/path/to/your/project')

# Import the functions from your updated file
from f1_data_service import get_race_strategies, fetch_strategies_fallback, fetch_session_data

# Set up cache directory
cache_dir = os.path.join(os.getcwd(), 'cache')
os.makedirs(cache_dir, exist_ok=True)
fastf1.Cache.enable_cache(cache_dir)

def test_race_strategy_extraction(year, round_number):
    """
    Test the extraction of race strategies for a specific race
    
    Parameters:
    - year: Season year (e.g., 2023)
    - round_number: Race round number
    
    Returns:
    - DataFrame with strategy data if successful
    """
    print(f"Testing strategy extraction for {year} Round {round_number}")
    
    # 1. Fetch the race session
    print("Fetching race session data...")
    race_session = fetch_session_data(year, round_number, 'R')
    
    if race_session is None:
        print("Failed to fetch race session data")
        return None
    
    # 2. Try to load full lap data with telemetry
    print("Loading lap data with telemetry...")
    try:
        race_session.load_laps(with_telemetry=True)
        print("Successfully loaded lap data with telemetry")
    except Exception as e:
        print(f"Could not load detailed lap data: {e}")
        print("Continuing with basic lap data...")
    
    # 3. Extract race strategies using the primary method
    print("Extracting race strategies using primary method...")
    strategies_df = get_race_strategies(race_session)
    
    if strategies_df is not None and not strategies_df.empty:
        print(f"SUCCESS: Extracted {len(strategies_df)} strategy records using primary method")
        print("\nSample of strategy data:")
        print(strategies_df.head())
        print("\nStrategy data summary:")
        print(strategies_df.describe())
        
        # Check if we have data for all drivers
        unique_drivers = strategies_df['driver_id'].nunique()
        print(f"\nStrategy data covers {unique_drivers} drivers")
        
        return strategies_df
    else:
        print("Primary method didn't return any strategy data")
    
    # 4. Try the fallback method
    print("\nTrying fallback method for strategy extraction...")
    fallback_strategies_df = fetch_strategies_fallback(race_session)
    
    if fallback_strategies_df is not None and not fallback_strategies_df.empty:
        print(f"SUCCESS: Extracted {len(fallback_strategies_df)} strategy records using fallback method")
        print("\nSample of fallback strategy data:")
        print(fallback_strategies_df.head())
        return fallback_strategies_df
    else:
        print("FAILURE: Could not extract any strategy data with either method")
        return None

def run_multiple_tests():
    """Run tests on multiple races to compare results"""
    test_races = [
        (2023, 1),  # Bahrain 2023
        (2023, 14), # Belgium 2023
        (2022, 5),  # Miami 2022
        (2021, 10), # Britain 2021
    ]
    
    results = {}
    
    for year, round_num in test_races:
        print("\n" + "="*50)
        print(f"TESTING {year} ROUND {round_num}")
        print("="*50)
        
        result = test_race_strategy_extraction(year, round_num)
        
        if result is not None:
            success = True
            count = len(result)
        else:
            success = False
            count = 0
            
        results[(year, round_num)] = {
            "success": success,
            "record_count": count
        }
        
    # Print summary of all tests
    print("\n\n" + "="*50)
    print("TEST SUMMARY")
    print("="*50)
    
    for (year, round_num), result in results.items():
        status = "SUCCESS" if result["success"] else "FAILURE"
        print(f"{year} Round {round_num}: {status} - {result['record_count']} records")

# Run the test
if __name__ == "__main__":
    print("Starting race strategy extraction tests\n")
    
    # Option 1: Test a specific race
    # test_race_strategy_extraction(2023, 1)  # 2023 Bahrain Grand Prix
    
    # Option 2: Run tests on multiple races
    run_multiple_tests()

ModuleNotFoundError: No module named 'f1_data_service'